# Predictive modeling and Robustness

This notebook investigates three machine learning architectures for predicting 30-day hospital readmission (`readmitted_binary`) and then evaluates how their predictions change under attacks targeting different stages of the machine-learning pipeline.

The experimental workflow is divided into two main stages:

1. **Model Experimentation & Hyperparameter Tuning**
   - train and optimize a Decision Tree, XGBoost classifier, and MLP;
   - assess cross-validation stability;
   - select decision thresholds using out-of-fold predictions;
   - perform a final evaluation on the test set.

2. **Robustness Benchmark**
   - evaluate **data poisoning**, **model-artifact poisoning**, and **inference-time evasion attacks**;
   - restrict evasion perturbations to feasible continuous features;
   - quantify both predictive degradation and attack success;
   - compare five defense strategies under the appropriate threat model;
   - summarize the main robustness trade-offs and worst-case vulnerabilities.

## 1. Experimental Objective

The first stage compares three complementary model families:

* **Decision Tree (`DecisionTreeClassifier`)**: A highly interpretable tree-based classifier capable of modelling non-linear decision rules through recursive feature splits.
* **XGBoost (`XGBClassifier`)**: A gradient-boosted tree ensemble that combines multiple shallow decision trees to model complex non-linear relationships and feature interactions.
* **Neural Network MLP (`MLPClassifier`)**: A feed-forward neural network capable of learning smooth non-linear decision boundaries.

Because the positive class represents 30-day readmission, recall is used as the primary model-selection criterion. The objective is to maximize the identification of positive cases.

In [1]:
import os
import random
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import (
    GridSearchCV, RandomizedSearchCV, StratifiedKFold,
    cross_val_score, cross_val_predict
)
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.base import clone
from sklearn.metrics import (
    recall_score, precision_score, f1_score, fbeta_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
import joblib
import warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# Fix all sources of randomness for reproducibility
SEED = 42

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(SEED)
print(f"Random seed fixed to: {SEED}")

Random seed fixed to: 42


## 2. Load the Processed Data

In [3]:
X_train = joblib.load(PROCESSED_DIR / "X_train.joblib")
X_test = joblib.load(PROCESSED_DIR / "X_test.joblib")
y_train = joblib.load(PROCESSED_DIR / "y_train.joblib")
y_test = joblib.load(PROCESSED_DIR / "y_test.joblib")

print(f"X_train: {X_train.shape[0]:,} rows x {X_train.shape[1]} features")
print(f"X_test:  {X_test.shape[0]:,} rows x {X_test.shape[1]} features")
print(f"y_train: {y_train.shape[0]:,} samples | positive_samples: {y_train.mean():.2%}")
print(f"y_test: {y_test.shape[0]:,} samples | positive_samples: {y_test.mean():.2%}")

X_train: 57,204 rows x 148 features
X_test:  14,302 rows x 148 features
y_train: 57,204 samples | positive_samples: 8.80%
y_test: 14,302 samples | positive_samples: 8.80%


## 3. Cross-Validation Strategy Class Imbalance and Hyperparameter Optimization

A 5-fold **StratifiedKFold** strategy is adopted for hyperparameter optimization. Stratification preserves approximately the same class distribution across folds, which is particularly important for this imbalanced binary classification problem.

Class imbalance is handled differently according to the model family:

- the Decision Tree explicitly searches over `class_weight`;
- XGBoost uses `scale_pos_weight`;
- the MLP relies on the chosen threshold to control the recall–precision trade-off.

The Decision Tree and XGBoost configurations are evaluated exhaustively using `GridSearchCV`. The MLP has a substantially larger search space, so `RandomizedSearchCV` is used to sample 250 configurations.

All searches optimize **recall** using the same stratified five-fold cross-validation strategy.

In [ ]:
cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scale_pos_weight = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)

model_grids = {
    "Decision Tree": (
        DecisionTreeClassifier(random_state=SEED, class_weight="balanced"),
        {
            "max_depth": [4, 6, 8, 10, 12, 15, 20, None],
            "min_samples_split": [2, 5, 10, 20, 40, 80],
            "min_samples_leaf": [1, 2, 5, 10, 20, 30, 50],
            "criterion": ["gini", "entropy", "log_loss"],
            "class_weight": [None, "balanced"]
        },
    ),
    "XGBoost": (
        XGBClassifier(
            objective="binary:logistic", eval_metric="logloss", tree_method="hist",
            max_bin=64, n_jobs=1, random_state=SEED,
            scale_pos_weight=scale_pos_weight
        ),
        {
            "n_estimators": [100, 200],
            "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.07, 0.1],
            "max_depth": [2, 3, 4, 5, 6, 8],
            "subsample": [0.65, 0.75, 0.85, 0.95, 1.0],
            "colsample_bytree": [0.6, 0.75, 0.9, 1.0],
        },
    ),
    "MLP": (
        MLPClassifier(max_iter=500, random_state=SEED, early_stopping=True),
        {
            "hidden_layer_sizes": [(32,),(64,),(128,),(64, 32),(128, 64),(128, 64, 32),(256, 128),(256, 128, 64)],
            "activation": ["relu", "tanh"],
            "alpha": [1e-5,1e-4,1e-3,1e-2,1e-1],
            "learning_rate_init": [1e-4,3e-4,1e-3,3e-3],
            "batch_size": [32,64,128,256],
            "solver": ["adam"]
        },
    ),
}

best_models = {}
grid_rows = []
for name, (estimator, param_grid) in model_grids.items():
    if(name != "MLP"):
        print(f"\n--- TRAINING / GRID SEARCH: {name} ---")
        search = GridSearchCV(
            estimator=estimator,
            param_grid=param_grid,
            scoring="recall",
            cv=cv_strat,
            n_jobs=2,
            refit=True,
        )
    else:
        print(f"\n--- TRAINING / RANDOM SEARCH: {name} ---")
        search = RandomizedSearchCV(
            estimator=estimator,
            param_distributions=param_grid,
            n_iter=250,
            scoring="recall",
            cv=cv_strat,
            random_state=SEED,
            n_jobs=2,
            refit=True
        )
        
    search.fit(X_train, y_train)
    best_models[name] = search.best_estimator_
    grid_rows.append({
        "Model": name,
        "Best CV Recall": search.best_score_,
        "Best Parameters": search.best_params_,
    })
    print(f"Best StratifiedKFold Recall: {search.best_score_:.4f}")
    print(f"Best parameters: {search.best_params_}")

grid_results_df = pd.DataFrame(grid_rows).sort_values("Best CV Recall", ascending=False).reset_index(drop=True)
print("\n=== GRID SEARCH TRAINING RESULTS ===")
display(grid_results_df.style.format({"Best CV Recall":"{:.4f}"}))


--- TRAINING / GRID SEARCH: Decision Tree ---
Best StratifiedKFold Recall: 0.6559
Best parameters: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 6, 'min_samples_leaf': 10, 'min_samples_split': 2}

--- TRAINING / GRID SEARCH: XGBoost ---
Best StratifiedKFold Recall: 0.6247
Best parameters: {'colsample_bytree': 1.0, 'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 100, 'subsample': 1.0}

--- TRAINING / RANDOM SEARCH: MLP ---
Best StratifiedKFold Recall: 0.0127
Best parameters: {'solver': 'adam', 'learning_rate_init': 0.003, 'hidden_layer_sizes': (256, 128, 64), 'batch_size': 64, 'alpha': 0.01, 'activation': 'relu'}

=== GRID SEARCH TRAINING RESULTS ===


,Model,Best CV Recall,Best Parameters
0,Decision Tree,0.6559,"{'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 6, 'min_samples_leaf': 10, 'min_samples_split': 2}"
1,XGBoost,0.6247,"{'colsample_bytree': 1.0, 'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 100, 'subsample': 1.0}"
2,MLP,0.0127,"{'solver': 'adam', 'learning_rate_init': 0.003, 'hidden_layer_sizes': (256, 128, 64), 'batch_size': 64, 'alpha': 0.01, 'activation': 'relu'}"


The **Decision Tree** obtains the highest mean cross-validated recall (**0.6559**), followed by **XGBoost (0.6247)**. Both substantially outperform the **MLP (0.0127)**.


## 4. Cross-Validation Stability Assessment

A second stratified five-fold evaluation is used to assess how consistently the tuned models perform across different folds.

The mean recall estimates average performance, while the standard deviation quantifies sensitivity to the particular train/validation partition.

A model with a slightly lower mean recall but substantially lower variability may be preferable when stability is an important requirement.

In [6]:
cv_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
stability_rows = []

for name, model in best_models.items():
    scores = cross_val_score(clone(model), X_train, y_train, cv=cv_kfold, scoring="recall", n_jobs=-1)
    stability_rows.append({
        "Model": name,
        "KFold Mean Recall": scores.mean(),
        "KFold Std": scores.std(),
        "Fold Recalls": np.round(scores, 4).tolist(),
    })
stability_results_df = pd.DataFrame(stability_rows).sort_values("KFold Mean Recall", ascending=False).reset_index(drop=True)
print(f"\n=== Stratified Cross-Validation Stability ===")
display(stability_results_df.style.format({"KFold Mean Recall":"{:.4f}","KFold Std":"{:.4f}"}))


=== Stratified Cross-Validation Stability ===


,Model,KFold Mean Recall,KFold Std,Fold Recalls
0,Decision Tree,0.6559,0.0215,"[0.6561, 0.6753, 0.6653, 0.6147, 0.668]"
1,XGBoost,0.6247,0.0234,"[0.6113, 0.6584, 0.6356, 0.5889, 0.6292]"
2,MLP,0.0127,0.0096,"[0.0199, 0.0218, 0.0, 0.0199, 0.002]"


The **Decision Tree** maintains a mean recall of **0.6559** with a standard deviation of **0.0215**; its fold recalls range from **0.6147 to 0.6753**. **XGBoost** obtains **0.6247 ± 0.0234**, with fold recalls from **0.5889 to 0.6584**. These relatively small standard deviations indicate that the recall of both tree-based models is reasonably consistent across the five stratified partitions.

The **MLP** has a lower numerical standard deviation (**0.0096**), but its mean recall is only **0.0127**, including one fold with recall **0.0000**. A small standard deviation is therefore not evidence of a strong model in this case: the MLP is consistently operating at very low sensitivity.

## 5. Decision-Threshold Optimization

A probability threshold of 0.50 is not necessarily appropriate for an imbalanced medical classification problem.

Thresholds are therefore optimized using **out-of-fold (OOF) predictions** generated exclusively from the training set. For each candidate threshold, precision, recall, and F2-score are computed.

The selected threshold maximizes recall while satisfying a minimum precision constraint of 0.20. F2 is used as a tie-breaker because it assigns greater importance to recall than precision.

In [7]:
MIN_PRECISION = 0.20
F2_BETA = 2.0
BASE_THRESHOLD = 0.50

def compute_threshold_metrics(y_true, y_probs, thresholds):
    """
    Evaluates Precision, Recall, and F2-Score for a given array of decision thresholds.
    """
    rows=[]
    for t in thresholds:
        # Convert continuous probabilities into binary predictions (0 or 1) based on the threshold
        pred=(y_probs>=t).astype(int)
        rows.append({
            "Threshold":t,
            "Precision":precision_score(y_true,pred,zero_division=0),
            "Recall":recall_score(y_true,pred,zero_division=0),
            "F2":fbeta_score(y_true,pred,beta=F2_BETA,zero_division=0),
        })
    return pd.DataFrame(rows)

def find_oof_threshold(model, X_train, y_train, cv_strategy, min_precision):
    """
    Finds the optimal threshold using Out-Of-Fold (OOF) cross-validation predictions 
    to maximize Recall while satisfying a minimum Precision constraint.
    """
    # Generate Out-Of-Fold probability predictions for the positive class (class 1)
    oof_probs = cross_val_predict(clone(model), X_train, y_train, cv=cv_strategy, method="predict_proba", n_jobs=-1)[:,1]
    # Generate a fine-grained grid of candidate thresholds from 0.01 to 0.95
    lower_range = np.linspace(0.01, 0.50, 100)
    upper_range = np.linspace(0.50, 0.95, 91)
    candidate_thresholds = np.unique(np.concatenate([lower_range, upper_range]))
    # Compute metrics for all candidate thresholds
    metrics_df = compute_threshold_metrics(y_train, oof_probs, candidate_thresholds)
    # Filter thresholds that satisfy the minimum Precision constraint
    eligible_thresholds = metrics_df[metrics_df["Precision"] >= min_precision]
    if eligible_thresholds.empty:
        best_row = metrics_df.loc[metrics_df["F2"].idxmax()]
    else:
        # Get threshold(s) with highest Recall, using F2 as a tie-breaker
        max_recall = eligible_thresholds["Recall"].max()
        best_row = (
            eligible_thresholds[eligible_thresholds["Recall"] == max_recall]
            .sort_values(by="F2", ascending=False)
            .iloc[0]
        )
    return (float(best_row["Threshold"]), float(best_row["Precision"]), float(best_row["Recall"]), float(best_row["F2"]))

threshold_results = []
optimal_thresholds = {}
for name,model in best_models.items():
    best_thresh, prec, rec, f2 = find_oof_threshold(
        model = model,
        X_train = X_train,
        y_train = y_train,
        cv_strategy = cv_strat,
        min_precision = MIN_PRECISION,
    )
    optimal_thresholds[name] = best_thresh
    threshold_results.append(
        {
            "Model": name,
            "OOF Threshold": best_thresh,
            "OOF Precision": prec,
            "OOF Recall": rec,
            "OOF F2": f2,
        }
    )
threshold_summary_df = (pd.DataFrame(threshold_results).sort_values(by="OOF Recall", ascending=False).reset_index(drop=True))   
print(f"\n=== Optimal OOF Thresholds for Maximum Recall (Min. Precision = {MIN_PRECISION}) ===")
display(threshold_summary_df.style.format({"OOF Threshold":"{:.3f}","OOF Precision":"{:.4f}","OOF Recall":"{:.4f}","OOF F2":"{:.4f}"}))


=== Optimal OOF Thresholds for Maximum Recall (Min. Precision = 0.2) ===


,Model,OOF Threshold,OOF Precision,OOF Recall,OOF F2
0,MLP,0.168,0.2011,0.2295,0.2232
1,XGBoost,0.565,0.2015,0.2180,0.2145
2,Decision Tree,0.645,0.2072,0.1856,0.1895


The automatic OOF rule selects markedly different operating points across the three models. The **MLP threshold is reduced to 0.168**, producing **0.2295 recall** at **0.2011 precision** and an F2-score of **0.2232**. This confirms that the MLP's extremely low recall at its default operating point is strongly threshold-dependent.

For the tree-based models, satisfying the 0.20 precision constraint requires thresholds above 0.50: **0.565 for XGBoost** and **0.645 for the Decision Tree**. Their corresponding OOF recalls fall to **0.2180** and **0.1856**, respectively. The selected points sit very close to the minimum allowed precision (**0.2015** and **0.2072**), showing the cost of enforcing that precision floor.

Because the main experimental objective is recall, we retains the default **0.50 threshold for Decision Tree and XGBoost**, while keeping **0.168 for the MLP**.

In [8]:
# Update the dictionary with the default threshold
optimal_thresholds["Decision Tree"] = BASE_THRESHOLD
optimal_thresholds["XGBoost"] = BASE_THRESHOLD

print(optimal_thresholds)

{'Decision Tree': 0.5, 'XGBoost': 0.5, 'MLP': 0.16838383838383839}


## 6. Final Evaluation on the Test Set

The tuned models are now evaluated on the unseen test set using the thresholds selected from the previous section.

Threshold-dependent metrics include recall, precision, F1, F2, and confusion-matrix counts. **ROC-AUC** and **PR-AUC** instead evaluate ranking quality over all possible thresholds.

The test set is used only for final reporting and is not used to modify the selected models or thresholds.

In [9]:
def evaluate_model(model, threshold, X_test, y_test):
    """
    Evaluates classification performance using a specified decision threshold
    and returns a dictionary of threshold-dependent and independent metrics.
    """
    probs = model.predict_proba(X_test)[:,1]
    preds = (probs>=threshold).astype(int)
    tn,fp,fn,tp = confusion_matrix(y_test,preds,labels=[0, 1]).ravel()
    return {
        "Recall":recall_score(y_test,preds,zero_division=0),
        "Precision":precision_score(y_test,preds,zero_division=0),
        "F1":f1_score(y_test,preds,zero_division=0),
        "F2":fbeta_score(y_test,preds,beta=F2_BETA,zero_division=0),
        "ROC-AUC":roc_auc_score(y_test,probs),
        "PR-AUC":average_precision_score(y_test,probs),
        "TN":tn,"FP":fp,"FN":fn,"TP":tp,
    }

evaluation_results = []

for order, name in enumerate(model_grids.keys(),start=1):
    model_instance = best_models[name]
    optimal_thresh = optimal_thresholds[name]
    optimal_metrics = evaluate_model(model=model_instance,threshold=optimal_thresh,X_test=X_test,y_test=y_test)
    evaluation_results.append(
        {
            "Order": order,
            "Model": name,
            "Threshold": optimal_thresh,
            **optimal_metrics,
        }
    )
test_results_df = pd.DataFrame(evaluation_results).sort_values(["Recall","F2"],ascending=False).reset_index(drop=True)
print(f"\n=== Final Test Set Evaluation with selected thresholds ===")
display(test_results_df.style.format({"Threshold":"{:.3f}","Recall":"{:.4f}","Precision":"{:.4f}","F1":"{:.4f}","F2":"{:.4f}","ROC-AUC":"{:.4f}","PR-AUC":"{:.4f}"}))


=== Final Test Set Evaluation with selected thresholds ===


,Order,Model,Threshold,Recall,Precision,F1,F2,ROC-AUC,PR-AUC,TN,FP,FN,TP
0,1,Decision Tree,0.500,0.6479,0.1170,0.1982,0.3397,0.6395,0.1536,6895,6149,443,815
1,2,XGBoost,0.500,0.6081,0.1223,0.2037,0.3389,0.6433,0.1639,7556,5488,493,765
2,3,MLP,0.168,0.1017,0.2645,0.1470,0.1160,0.6485,0.1621,12688,356,1130,128


On the test set, the **Decision Tree provides the highest recall, 0.6479**, identifying **815 of 1,258** positive cases. XGBoost follows with **0.6081 recall** (**765 true positives**). Their F2-scores are almost identical (**0.3397** for the Decision Tree and **0.3389** for XGBoost), indicating a very similar overall recall-weighted precision/recall trade-off.

The Decision Tree pays for its higher sensitivity with more false positives (**6,149**) than XGBoost (**5,488**), and therefore has slightly lower precision (**0.1170 vs. 0.1223**). XGBoost also has the better ranking metrics of the two tree models: **ROC-AUC 0.6433** and **PR-AUC 0.1639**, compared with **0.6395** and **0.1536** for the Decision Tree.

The **MLP** has the highest precision (**0.2645**) and the highest ROC-AUC (**0.6485**), with PR-AUC (**0.1621**) close to XGBoost. However, at the retained threshold of 0.168 it reaches only **0.1017 recall**, detecting **128 of 1,258** positives. This combination suggests that its probability ranking contains signal, but the selected operating point does not provide the sensitivity required by the recall-oriented objective.

## 7. Save Final Models and Experimental Artifacts
The complete models, thresholds, and selected hyperparameters are saved locally.

In [10]:
# Save each final model separately
model_files = {
    "Decision Tree": "decision_tree_best.joblib",
    "XGBoost": "xgboost_best.joblib",
    "MLP": "mlp_best.joblib",
}

for model_name, filename in model_files.items():
    joblib.dump(
        best_models[model_name],
        os.path.join(MODEL_DIR, filename)
    )

# Save the optimized probability thresholds used for the final test predictions
joblib.dump(
    {name: float(optimal_thresholds[name]) for name in optimal_thresholds},
    os.path.join(MODEL_DIR, "optimal_thresholds.joblib")
)

# Save the selected the best hyperparameters for each model
best_hyperparameters = {
    name: best_models[name].get_params()
    for name in best_models
}

joblib.dump(
    best_hyperparameters,
    os.path.join(MODEL_DIR, "best_hyperparameters.joblib")
)

print(f"Saved final models and artifacts to: {MODEL_DIR}")
for filename in [*model_files.values(), "optimal_thresholds.joblib", "best_hyperparameters.joblib"]:
    print(" -", os.path.join(MODEL_DIR, filename))

Saved final models and artifacts to: C:\Users\lucad\Desktop\Master\2° Year\Ethics in Artificial Intelligence\Module 2\Project\models
 - C:\Users\lucad\Desktop\Master\2° Year\Ethics in Artificial Intelligence\Module 2\Project\models\decision_tree_best.joblib
 - C:\Users\lucad\Desktop\Master\2° Year\Ethics in Artificial Intelligence\Module 2\Project\models\xgboost_best.joblib
 - C:\Users\lucad\Desktop\Master\2° Year\Ethics in Artificial Intelligence\Module 2\Project\models\mlp_best.joblib
 - C:\Users\lucad\Desktop\Master\2° Year\Ethics in Artificial Intelligence\Module 2\Project\models\optimal_thresholds.joblib
 - C:\Users\lucad\Desktop\Master\2° Year\Ethics in Artificial Intelligence\Module 2\Project\models\best_hyperparameters.joblib


# Robustness Benchmark

The second stage evaluates whether the selected models remain reliable under **three distinct threat families**, ordered according to where they act in the machine-learning lifecycle:

1. **Data poisoning** — the attacker corrupts the training labels before fitting.
2. **Model poisoning** — the attacker tampers with the fitted model artifact.
3. **Adversarial/evasion attacks** — the attacker perturbs inputs at inference time.

First we test attacks against the training process and model artifact, then attacks against already-deployed predictors.

For evasion attacks, only continuous numerical variables are perturbable; binary and one-hot encoded features remain frozen. For poisoning attacks, the test inputs are left unchanged because the compromise occurs in the labels or model parameters rather than in the inference samples.

## 1. Robustness Setup

The helper module centralizes attack generation, poisoning utilities, defense transformations, and the unified defense-comparison logic.

The same fixed evaluation subset is reused for poisoning-defense comparisons so that changes across attacks and defenses are not caused by different test samples.


In [11]:
from src.robustness.robustness_helper import (
    identify_perturbable_columns,
    clean_correct_subset,
    tabular_score_attack,
    run_art_attack,
    attack_statistics,
    evaluate_probability_output,
    feature_squeeze,
    evaluate_outlier_rejection,
    randomized_smoothing_predict_proba,
    adversarial_train,
    build_ensemble_adversarial_dataset,
    poison_training_labels,
    train_with_data_poisoning,
    poison_model_artifact,
    build_defense_comparison_matrix,
)
from sklearn.ensemble import IsolationForest

# Number of test samples used by the attack benchmark.
ATTACK_SAMPLE_SIZE = min(500, len(X_test))

# Number of training samples used to generate adversarial examples.
AT_TRAIN_SAMPLE_SIZE = min(5000, len(X_train))

# Fixed test subset reused by poisoning-defense experiments.
X_POISON_EVAL = X_test.iloc[:ATTACK_SAMPLE_SIZE].copy()
y_POISON_EVAL = np.asarray(y_test)[:ATTACK_SAMPLE_SIZE].astype(int)

## 2. Data Poisoning

Labels in the training set are maliciously flipped and the model is retrained. We test both random label flips and targeted `positive → negative` flips, the latter directly attacking recall of hospital readmissions.

For both strategies, three poisoning fractions are evaluated: `1%`, `5%`, and `10%`.

For random label flipping, the fraction refers to the **entire training set**. For targeted positive-to-negative poisoning, it refers only to the **positive training samples**. Therefore, a 10% targeted poisoning attack flips 10% of the available `readmitted` labels rather than 10% of all training observations.

For each poisoning configuration, a copy of the model is retrained on the corrupted labels while keeping the selected hyperparameters and decision threshold unchanged. This makes it possible to isolate the effect of training-label corruption from changes in model configuration.

In [12]:
poisoning_clean_baseline = {}
for model_name, model in best_models.items():
    threshold = float(optimal_thresholds[model_name])
    poisoning_clean_baseline[model_name] = evaluate_model(
        model,
        threshold,
        X_test,
        y_test,
    )

DATA_POISON_FRACTIONS = [0.01, 0.05, 0.10]
DATA_POISON_STRATEGIES = [
    ("Random label flip", "random"),
    ("Positive → negative", "positive_to_negative"),
]

data_poisoning_rows = []
data_poisoning_examples = {}

for model_name, model in best_models.items():
    threshold = float(optimal_thresholds[model_name])
    clean_recall = poisoning_clean_baseline[model_name]["Recall"]

    for strategy_idx, (strategy_label, strategy_key) in enumerate(
        DATA_POISON_STRATEGIES
    ):
        for fraction in DATA_POISON_FRACTIONS:
            poison_seed = (
                SEED
                + 100 * strategy_idx
                + int(fraction * 10000)
            )

            poisoned_model, y_poisoned, poisoned_idx = (
                train_with_data_poisoning(
                    model=model,
                    X_train=X_train,
                    y_train=y_train,
                    fraction=fraction,
                    strategy=strategy_key,
                    seed=poison_seed,
                )
            )

            poisoned_metrics = evaluate_model(
                poisoned_model,
                threshold,
                X_test,
                y_test,
            )

            attack_label = (
                f"{strategy_label} / {fraction:.0%}"
            )

            data_poisoning_rows.append(
                {
                    "Model": model_name,
                    "Poisoning Strategy": strategy_label,
                    "Poison Fraction": fraction,
                    "Actual Poisoned Samples": len(poisoned_idx),
                    "Test Recall": poisoned_metrics["Recall"],
                    "Recall Drop": (
                        clean_recall
                        - poisoned_metrics["Recall"]
                    ),
                    "Test Precision": poisoned_metrics["Precision"],
                    "Test F2": poisoned_metrics["F2"],
                    "Test PR-AUC": poisoned_metrics["PR-AUC"],
                }
            )

            # Keep the compromised model and attack metadata for the
            # unified defense comparison later in the notebook.
            data_poisoning_examples[(model_name, attack_label)] = {
                "attacked_model": poisoned_model,
                "X_eval": X_POISON_EVAL.copy(),
                "y_eval": y_POISON_EVAL.copy(),
                "fraction": fraction,
                "strategy_key": strategy_key,
                "strategy_label": strategy_label,
                "seed": poison_seed,
            }

data_poisoning_df = (
    pd.DataFrame(data_poisoning_rows)
    .sort_values(
        ["Model", "Poisoning Strategy", "Poison Fraction"]
    )
    .reset_index(drop=True)
)

display(
    data_poisoning_df.style
    .hide(axis="index")
    .format(
        {
            "Poison Fraction": "{:.1%}",
            "Test Recall": "{:.4f}",
            "Recall Drop": "{:.4f}",
            "Test Precision": "{:.4f}",
            "Test F2": "{:.4f}",
            "Test PR-AUC": "{:.4f}",
        }
    )
)

Model,Poisoning Strategy,Poison Fraction,Actual Poisoned Samples,Test Recall,Recall Drop,Test Precision,Test F2,Test PR-AUC
Decision Tree,Positive → negative,1.0%,50,0.6343,0.0135,0.1196,0.3409,0.1525
Decision Tree,Positive → negative,5.0%,252,0.6439,0.0040,0.1177,0.3399,0.1564
Decision Tree,Positive → negative,10.0%,503,0.6797,-0.0318,0.1166,0.3458,0.1518
Decision Tree,Random label flip,1.0%,572,0.6502,-0.0024,0.1155,0.3377,0.1502
Decision Tree,Random label flip,5.0%,2860,0.6002,0.0477,0.1243,0.3399,0.1465
Decision Tree,Random label flip,10.0%,5720,0.5978,0.0501,0.1198,0.3325,0.1469
MLP,Positive → negative,1.0%,50,0.2536,-0.1518,0.2073,0.2427,0.1767
MLP,Positive → negative,5.0%,252,0.0350,0.0668,0.2914,0.0424,0.1593
MLP,Positive → negative,10.0%,503,0.2774,-0.1757,0.1911,0.2544,0.1618
MLP,Random label flip,1.0%,572,0.2027,-0.1010,0.1992,0.2020,0.1610


The effect of label poisoning is strongly model-dependent and is not always monotonic with the poisoning fraction.

For the **Decision Tree**, random label flipping causes the clearest degradation: recall falls from the clean value of **0.6479** to **0.6002** at 5% poisoning and **0.5978** at 10% poisoning. Targeted positive-to-negative poisoning is less monotonic: recall is **0.6343** at 1%, **0.6439** at 5%, and rises to **0.6797** at 10%. This increase should not be interpreted as the attack improving the model; retraining on altered labels can change tree splits and the fixed decision boundary in non-monotonic ways.

**XGBoost** is substantially affected by targeted poisoning at the highest level: 10% positive-to-negative flips reduce recall from **0.6081 to 0.4253**. In contrast, 5% and 10% random label flipping drive recall to **0.9928** and **0.9984**, but precision simultaneously collapses to **0.0892** and **0.0880**. The model is therefore not becoming robust; it is moving toward an excessively positive prediction regime. The nearly unchanged PR-AUC values around **0.163** reinforce that this behavior is largely an operating-point/calibration shift rather than a genuine improvement in ranking quality.

The **MLP** is the most unstable under retraining. Targeted 5% poisoning reduces recall to **0.0350**, whereas the 1% and 10% configurations produce recalls of **0.2536** and **0.2774**, both above the clean value of 0.1017. Random flips also increase recall while generally reducing or redistributing precision. This irregular response is consistent with a stochastic non-convex estimator evaluated at a threshold selected on the clean model: poisoning can change calibration and the number of positive predictions in either direction.

The main conclusion is therefore not that larger label corruption always produces larger recall loss. **Data poisoning changes both the learned decision function and its calibration**, so recall must be interpreted together with precision, F2, and PR-AUC.

## 3. Model Poisoning

Because **Decision Tree**, **XGBoost**, and **MLP** have different internal representations, the poisoning procedure is model-specific.

### Decision Tree

For the Decision Tree, the weighted value associated with the positive class in each tree node is reduced according to:

$$positive\_mass_{poisoned} = positive\_mass_{clean} \cdot (1 - s)$$

where $s$ represents the poisoning severity.

### MLP

For the MLP, the bias of the final output layer is directly reduced:

$$b_{poisoned} = b_{clean} - s$$

This shifts the model output toward the negative class and consequently lowers the predicted probability of `readmitted`. The learned network weights and the remaining layers are left unchanged.

### XGBoost

For XGBoost, the model's `base_score` is reduced according to:

$$base\_score_{poisoned} = base\_score_{clean} \cdot (1 - s)$$

The `base_score` represents the initial prediction level of the model. Lowering it introduces a systematic bias toward class `0`, while the learned boosted trees themselves remain unchanged.


The experiment evaluates three severity levels: `0.10`, `0.25`, and `0.50`. Increasing the severity produces a stronger suppression of positive-class predictions. However, severity does **not** represent exactly the same mathematical amount of corruption for all three models, because it acts on different internal parameters. Its common interpretation is therefore the **strength of the model-artifact manipulation directed toward reducing positive-class detection**.

After poisoning, each modified model is evaluated on the unchanged test set using the same decision threshold as the corresponding clean model. This allows the effect of model tampering to be measured without introducing changes in the input data, selected hyperparameters, or decision threshold.

In [13]:
MODEL_POISON_SEVERITIES = [0.10, 0.25, 0.50]

model_poisoning_rows = []
model_poisoning_examples = {}

for model_name, model in best_models.items():
    threshold = float(optimal_thresholds[model_name])
    clean_recall = poisoning_clean_baseline[model_name]["Recall"]

    for severity in MODEL_POISON_SEVERITIES:
        poisoned_model = poison_model_artifact(
            model=model,
            model_name=model_name,
            severity=severity,
        )

        poisoned_metrics = evaluate_model(
            poisoned_model,
            threshold,
            X_test,
            y_test,
        )

        attack_label = (
            f"Artifact tampering / severity={severity:.2f}"
        )

        model_poisoning_rows.append(
            {
                "Model": model_name,
                "Severity": severity,
                "Test Recall": poisoned_metrics["Recall"],
                "Recall Drop": (
                    clean_recall
                    - poisoned_metrics["Recall"]
                ),
                "Test Precision": poisoned_metrics["Precision"],
                "Test F2": poisoned_metrics["F2"],
                "Test PR-AUC": poisoned_metrics["PR-AUC"],
            }
        )

        model_poisoning_examples[(model_name, attack_label)] = {
            "attacked_model": poisoned_model,
            "X_eval": X_POISON_EVAL.copy(),
            "y_eval": y_POISON_EVAL.copy(),
            "severity": severity,
        }

model_poisoning_df = (
    pd.DataFrame(model_poisoning_rows)
    .sort_values(["Model", "Severity"])
    .reset_index(drop=True)
)

display(
    model_poisoning_df.style
    .hide(axis="index")
    .format(
        {
            "Severity": "{:.2f}",
            "Test Recall": "{:.4f}",
            "Recall Drop": "{:.4f}",
            "Test Precision": "{:.4f}",
            "Test F2": "{:.4f}",
            "Test PR-AUC": "{:.4f}",
        }
    )
)


Model,Severity,Test Recall,Recall Drop,Test Precision,Test F2,Test PR-AUC
Decision Tree,0.10,0.4189,0.2289,0.1562,0.3135,0.1536
Decision Tree,0.25,0.1979,0.4499,0.2278,0.2033,0.1536
Decision Tree,0.50,0.0000,0.6479,0.0000,0.0000,0.1536
MLP,0.10,0.0763,0.0254,0.2712,0.0891,0.1621
MLP,0.25,0.0485,0.0533,0.2961,0.0582,0.1621
MLP,0.50,0.0127,0.0890,0.2807,0.0157,0.1621
XGBoost,0.10,0.2925,0.3156,0.1942,0.2656,0.1639
XGBoost,0.25,0.1097,0.4984,0.2604,0.1241,0.1639
XGBoost,0.50,0.0000,0.6081,0.0000,0.0000,0.1639


Model artifact poisoning produces a much more systematic degradation than data poisoning. Increasing severity progressively suppresses positive predictions for all three architectures.

For the **Decision Tree**, recall decreases from the clean **0.6479** to **0.4189** at severity 0.10 and **0.1979** at 0.25, before reaching **0.0000** at 0.50. **XGBoost** follows the same pattern, falling from **0.6081** to **0.2925**, **0.1097**, and finally **0.0000**. The **MLP** starts from a much lower clean recall and declines from **0.1017** to **0.0763**, **0.0485**, and **0.0127**.

A notable result is that **PR-AUC remains unchanged within each model across severities**: 0.1536 for the Decision Tree, 0.1639 for XGBoost, and 0.1621 for the MLP. This is consistent with the design of the poisoning operation, which mainly shifts the model away from predicting the positive class while largely preserving the ordering of examples by score. As a result, many observations cross the fixed classification threshold even though ranking quality changes little.

At severity **0.50**, both tree-based models lose all positive detections.

## 3. Adversarial / Evasion Attacks

| Attack | Access Model | Principle |
|---|---|---|
| **TabularScore** | Score-based black-box | Coordinate-ascent attack using prediction probabilities to increase the loss of the true class. |
| **DecisionTreeAttack** | White-box | Exploits the internal structure of decision trees. |
| **HopSkipJump** | Decision-based black-box | Searches for a nearby decision boundary using model decisions. |
| **BoundaryAttack** | Decision-based black-box | Explores the decision boundary while reducing perturbation distance. |

### Identifying Perturbable Features

Continuous variables are considered perturbable. Binary variables and one-hot encoded features are frozen because arbitrary modifications could create invalid categorical states.

The attack pipeline also clips perturbed continuous features to the empirical minimum and maximum observed in the training data.

In [14]:
ATTACK_NAMES = [
    "TabularScore",
    "DecisionTreeAttack",
    "HopSkipJump",
    "BoundaryAttack",
]

PERTURBABLE_COLS = identify_perturbable_columns(X_train)
PERTURBABLE_IDX = np.array([X_train.columns.get_loc(c) for c in PERTURBABLE_COLS],dtype=int)

TRAIN_MIN = X_train[PERTURBABLE_COLS].min().astype(float).to_numpy()
TRAIN_MAX = X_train[PERTURBABLE_COLS].max().astype(float).to_numpy()
TRAIN_STD = (
    X_train[PERTURBABLE_COLS]
    .std()
    .replace(0.0, 1.0)
    .astype(float)
    .to_numpy(dtype=np.float32, copy=True)
)

print(f"Perturbable features: {len(PERTURBABLE_COLS)}")
print(f"Frozen features: {X_train.shape[1] - len(PERTURBABLE_COLS)}")

Perturbable features: 24
Frozen features: 124


In [15]:
attack_results = []
attack_examples = {}

rng = np.random.default_rng(SEED + 10)

for model_name, model in best_models.items():
    threshold = float(optimal_thresholds[model_name])

    X_candidate = X_test.iloc[:ATTACK_SAMPLE_SIZE].copy()
    y_candidate = np.asarray(y_test)[:ATTACK_SAMPLE_SIZE].astype(int)

    X_clean, y_clean, _ = clean_correct_subset(model,threshold,X_candidate,y_candidate)

    print(f"\n=== {model_name} ===")
    print(f"Candidate samples: {len(X_candidate):,}")
    print(f"Clean-correct attack samples: {len(X_clean):,}")
    print(f"Clean-correct positives: {(y_clean == 1).sum():,}")

    for attack_name in ATTACK_NAMES:
        print(f"Running {attack_name} ...")
        if attack_name == "TabularScore":
            X_adv = tabular_score_attack(model=model,X=X_clean,y=y_clean,
                                         PERTURBABLE_IDX=PERTURBABLE_IDX,TRAIN_STD=TRAIN_STD,
                                         TRAIN_MIN=TRAIN_MIN,TRAIN_MAX=TRAIN_MAX)
        else:
            X_adv = run_art_attack(model_name=model_name,model=model,attack_name=attack_name,
                                   X_train=X_train,X_attack=X_clean,y_attack=y_clean,
                                   PERTURBABLE_IDX=PERTURBABLE_IDX,TRAIN_MIN=TRAIN_MIN,TRAIN_MAX=TRAIN_MAX)

        if X_adv is None:
            print(f"  -> Not applicable to {model_name}")
            continue

        metrics = evaluate_model(model,threshold,X_adv,y_clean)
        stats = attack_statistics(model,X_clean,X_adv,y_clean,threshold,PERTURBABLE_IDX)

        attack_results.append({
            "Model": model_name,
            "Attack": attack_name,
            "N": len(X_clean),
            **metrics,
            **stats,
        })

        attack_examples[(model_name, attack_name)] = {
            "X_clean": X_clean.copy(),
            "X_adv": X_adv.copy(),
            "y": y_clean.copy(),
        }


attack_results_df = pd.DataFrame(attack_results)
attack_results_df = (attack_results_df.sort_values(["Positive Attack Success","Attack Success Rate"],ascending=False,).reset_index(drop=True))

print(f"=== Adversarial Robustness Evaluation: Evasion Attacks ===")
display(
    attack_results_df.style.format({
        "Recall": "{:.4f}",
        "Precision": "{:.4f}",
        "F1": "{:.4f}",
        "F2": "{:.4f}",
        "ROC-AUC": "{:.4f}",
        "PR-AUC": "{:.4f}",
        "Attack Success Rate": "{:.4f}",
        "Positive Attack Success": "{:.4f}",
        "Mean Probability Change": "{:.4f}",
        "Mean |Probability Change|": "{:.4f}",
        "Mean L2": "{:.4f}",
        "Mean Linf": "{:.4f}",
        "Mean Changed Features": "{:.2f}",
    })
)


=== Decision Tree ===
Candidate samples: 500
Clean-correct attack samples: 265
Clean-correct positives: 27
Running TabularScore ...
Running DecisionTreeAttack ...
Running HopSkipJump ...
Running BoundaryAttack ...

=== XGBoost ===
Candidate samples: 500
Clean-correct attack samples: 292
Clean-correct positives: 28
Running TabularScore ...
Running DecisionTreeAttack ...
  -> Not applicable to XGBoost
Running HopSkipJump ...
Running BoundaryAttack ...

=== MLP ===
Candidate samples: 500
Clean-correct attack samples: 452
Clean-correct positives: 8
Running TabularScore ...
Running DecisionTreeAttack ...
  -> Not applicable to MLP
Running HopSkipJump ...
Running BoundaryAttack ...
=== Adversarial Robustness Evaluation: Evasion Attacks ===


,Model,Attack,N,Recall,Precision,F1,F2,ROC-AUC,PR-AUC,TN,FP,FN,TP,Clean Correct N,Clean Correct Positives,Successful Attacks,Successful Positive Attacks,Attack Success Rate,Positive Attack Success,Mean Probability Change,Mean |Probability Change|,Mean L2,Mean Linf,Mean Changed Features
0,MLP,TabularScore,452,0.7500,0.3750,0.5000,0.6250,0.9840,0.5923,434,10,2,6,452,8,12,2,0.0265,0.2500,0.0175,0.0182,0.8167,0.2500,18.32
1,Decision Tree,HopSkipJump,265,0.8519,0.1031,0.1840,0.3474,0.6338,0.4359,38,200,4,23,265,27,204,4,0.7698,0.1481,0.1236,0.1457,1.2421,1.0791,20.97
2,XGBoost,TabularScore,292,0.9286,1.0000,0.9630,0.9420,0.9539,0.9386,264,0,2,26,292,28,2,2,0.0068,0.0714,-0.0004,0.0029,0.0096,0.0092,0.20
3,Decision Tree,DecisionTreeAttack,265,0.9630,0.6341,0.7647,0.8725,0.9683,0.8629,223,15,1,26,265,27,16,1,0.0604,0.0370,-0.0437,0.1048,1.3627,1.2396,1.48
4,Decision Tree,BoundaryAttack,265,1.0000,0.3068,0.4696,0.6888,0.9230,0.7724,177,61,0,27,265,27,61,0,0.2302,0.0000,0.0321,0.0362,0.4137,0.2802,22.01
5,Decision Tree,TabularScore,265,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,238,0,0,27,265,27,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.00
6,XGBoost,HopSkipJump,292,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,264,0,0,28,292,28,0,0,0.0000,0.0000,-0.0009,0.0010,0.0000,0.0000,0.02
7,XGBoost,BoundaryAttack,292,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,264,0,0,28,292,28,0,0,0.0000,0.0000,-0.0015,0.0016,0.0000,0.0000,0.47
8,MLP,HopSkipJump,452,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,444,0,0,8,452,8,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.00
9,MLP,BoundaryAttack,452,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,444,0,0,8,452,8,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.00


The **Decision Tree is most vulnerable to HopSkipJump**, with an overall attack success rate of **0.7698**. However, its positive attack success is only **0.1481** and recall remains **0.8519**. The main damage is therefore caused by originally correct negative observations being pushed into the positive class, which explains the very low precision (**0.1031**) and the **200 false positives**. `BoundaryAttack` also changes many decisions (**0.2302 attack success**) but causes **no successful attacks on the 27 initially correct positives**, leaving recall at 1.0 on this attack subset. `DecisionTreeAttack` is milder, with **0.0604** overall success and one successful positive attack. `TabularScore` does not find a successful perturbation for the Decision Tree under the configured constraints.

**XGBoost is the most resistant model in this benchmark.** HopSkipJump and BoundaryAttack produce no successful decisions changes, while TabularScore succeeds on only **2 of 292** initially correct samples (**0.0068 attack success**) and on **2 of 28** initially correct positives (**0.0714 positive attack success**).

For the **MLP**, TabularScore has a low overall success rate (**0.0265**) but flips **2 of the 8** initially correct positive samples, giving a positive attack success of **0.2500**. This percentage should be interpreted cautiously because the denominator contains only eight positives. HopSkipJump and BoundaryAttack do not find successful adversarial examples under the current constraints.

Overall, **XGBoost shows the strongest evasion robustness**, whereas the Decision Tree is particularly exposed to decision-based boundary attacks. The difference between overall and positive-class attack success also shows why both metrics are necessary in a readmission task.


## 4. Defense Strategies


Five defenses are evaluated against the three threat families. Not every defense is expected to be equally suitable for every threat; the purpose of the unified experiment is to measure that limitation rather than assume effectiveness.

| Defense | Category | Main mechanism |
|---|---|---|
| **Feature Squeezing** | Input transformation | Reduces numerical precision to suppress fine-grained input perturbations. |
| **Isolation Forest Rejection** | Anomaly detection | Rejects samples that look out-of-distribution before classification. |
| **Randomized Smoothing** | Input transformation | Averages predictions over Gaussian perturbations of continuous features. |
| **Adversarial Training** | Model hardening | Retrains on clean data plus adversarial examples generated against the same model. |
| **Ensemble Adversarial Training** | Model hardening | Retrains on adversarial examples generated by the other model families. |

For **data poisoning**, the two model-level defenses are attacked by poisoning the labels of the actual augmented training set used by each defense and retraining the estimator. For **model-artifact poisoning**, the already-hardened fitted model is tampered with directly.

This design avoids attributing protection to a defense that was never exposed to the relevant attack.

### 4.1 Feature Squeezing

Feature squeezing rounds perturbable continuous variables to a lower numerical precision before classification.

For evasion, both the clean reference input and adversarial input are transformed before attack success is measured. For data or model poisoning, the same inference input is supplied to the intact and compromised models after squeezing. Any observed protection therefore reflects whether the input transformation can reduce the effect of the compromised predictor.

### 4.2 Isolation Forest

The Isolation Forest is fitted only on the clean training distribution and is used as a rejection mechanism at inference time.

For poisoning attacks, the test observations themselves are not modified, so anomaly rejection is not expected to detect the underlying compromise directly. It is nevertheless evaluated to make this limitation explicit.

Classification metrics for this defense are reported on **accepted samples only** and must therefore be interpreted together with `Coverage` and `Rejection Rate`.


In [16]:
outlier_detector = IsolationForest(
    n_estimators=300,
    contamination="auto",
    random_state=SEED,
    n_jobs=-1,
)

outlier_detector.fit(X_train[PERTURBABLE_COLS].astype(float))

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",300
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary <n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",'auto'
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary <warm_start>`... versionadded:: 0.21",False
Name,Type,Value
estimator_ estimator_: :class:`~sklearn.tree.ExtraTreeRegressor` instanceThe child estimator template used to create the collection offitted sub-estimators... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,ExtraTreeRegressor,ExtraTreeRegr...ndom_state=42)


### 4.3 Randomized Smoothing

Randomized smoothing evaluates each observation under multiple independent Gaussian perturbations of the continuous features and averages the resulting positive-class probabilities:

$$
\bar{p}(y=1\mid x)
=
\frac{1}{N}
\sum_{i=1}^{N}
p(y=1\mid x+\epsilon_i).
$$

For poisoning comparisons, the same random noise realization is used for the intact and compromised models. This reduces the risk of incorrectly attributing Monte Carlo variation to the poisoning attack.

### 4.4 Adversarial Training

Adversarial training augments the clean training set with `TabularScore` adversarial examples generated from correctly classified training observations and then fits a fresh estimator.

The augmented dataset is retained because the later data-poisoning experiment must attack the actual training set used by this defense rather than the original clean dataset.

In [18]:
adversarially_trained_models = {}
adversarial_training_sets = {}
adversarial_training_rows = []
for model_name, model in best_models.items():
    print(f"Training adversarially trained {model_name} ...")

    (robust_model,
     X_adv_train_aug,
     y_adv_train_aug,
    ) = adversarial_train(model_name,model,optimal_thresholds[model_name],X_train,y_train,PERTURBABLE_IDX,TRAIN_STD,TRAIN_MIN,TRAIN_MAX,AT_TRAIN_SAMPLE_SIZE,seed=SEED + 400, return_training_data=True)
    adversarially_trained_models[model_name] = robust_model
    adversarial_training_sets[model_name] = (
        X_adv_train_aug,
        y_adv_train_aug,
    )

    threshold = float(optimal_thresholds[model_name])
 
    adversarial_training_rows.append({
        "Model": model_name,
        "Condition": "Adversarial training / clean test",
        **evaluate_model(
            robust_model,
            threshold,
            X_test,
            y_test
        ),
    })

adversarial_training_df = pd.DataFrame(adversarial_training_rows)

display(
    adversarial_training_df.style.format({
        "Recall": "{:.4f}",
        "Precision": "{:.4f}",
        "F2": "{:.4f}",
        "PR-AUC": "{:.4f}",
    })
)

Training adversarially trained Decision Tree ...
Training adversarially trained XGBoost ...
Training adversarially trained MLP ...


,Model,Condition,Recall,Precision,F1,F2,ROC-AUC,PR-AUC,TN,FP,FN,TP
0,Decision Tree,Adversarial training / clean test,0.6502,0.1172,0.198544,0.3404,0.639258,0.1515,6880,6164,440,818
1,XGBoost,Adversarial training / clean test,0.6073,0.1228,0.204333,0.3395,0.643477,0.1635,7588,5456,494,764
2,MLP,Adversarial training / clean test,0.2011,0.2189,0.209611,0.2044,0.635820,0.1651,12141,903,1005,253


Adversarial training preserves the clean operating characteristics of the two tree-based models. The Decision Tree obtains **0.6502 recall**, compared with **0.6479** before adversarial training, while XGBoost obtains **0.6073**, compared with **0.6081**. Their PR-AUC values also remain close to the original results (**0.1515 vs. 0.1536** for the Decision Tree and **0.1635 vs. 0.1639** for XGBoost). This indicates that the additional adversarial samples do not impose a meaningful clean-performance penalty on these two models.

The **MLP** changes more substantially: recall increases from **0.1017 to 0.2011** and F2 from **0.1160 to 0.2044**, while precision decreases from **0.2645 to 0.2189**. PR-AUC increases slightly from **0.1621 to 0.1651**. The main effect is therefore a shift toward predicting more positives, improving sensitivity at the cost of precision.

These clean-test results are important because a defense that improves attack resistance only by severely damaging normal predictive performance would not be practically useful.

### 4.5 Ensemble Adversarial Training

Ensemble adversarial training augments each target model with adversarial examples generated by the other model families. The objective is to expose the estimator to a more diverse set of perturbation patterns and reduce over-specialization to attacks generated against itself.

As with standard adversarial training, the resulting augmented dataset is retained so that later data-poisoning experiments can compromise the defended training pipeline directly.

In [19]:
ensemble_adversarial_models = {}
ensemble_adversarial_training_sets = {}

for model_name, target_model in best_models.items():
    print(f"Training ensemble-adversarial {model_name} ...")

    X_aug, y_aug = build_ensemble_adversarial_dataset(
        target_name=model_name,
        X_train=X_train,
        y_train=y_train,
        best_models=best_models,
        optimal_thresholds=optimal_thresholds,
        PERTURBABLE_IDX=PERTURBABLE_IDX,
        TRAIN_STD=TRAIN_STD,
        TRAIN_MIN=TRAIN_MIN,
        TRAIN_MAX=TRAIN_MAX,
        AT_TRAIN_SAMPLE_SIZE=AT_TRAIN_SAMPLE_SIZE,
        seed=SEED + 500,
    )

    robust_model = clone(target_model)
    robust_model.fit(X_aug, y_aug)

    ensemble_adversarial_models[model_name] = robust_model
    ensemble_adversarial_training_sets[model_name] = (X_aug,y_aug)


Training ensemble-adversarial Decision Tree ...
Training ensemble-adversarial XGBoost ...
Training ensemble-adversarial MLP ...


## 5. Unified Defense Comparison Matrix

The matrix below is the central defense experiment and covers **evasion, data poisoning, and model-artifact poisoning**.

The comparison follows threat-specific rules:

- **Evasion + input defenses:** evaluate transformed adversarial inputs.
- **Evasion + model-level defenses:** generate a fresh adaptive `TabularScore` attack against the hardened model.
- **Data poisoning + input defenses:** compare an intact and poisoned model under the same input transformation.
- **Data poisoning + model-level defenses:** poison the labels of the defense-specific augmented training set, retrain, and compare with the corresponding intact hardened model.
- **Model poisoning + input defenses:** evaluate the tampered model under the input defense.
- **Model poisoning + model-level defenses:** tamper with the already-hardened fitted artifact.

For data and model poisoning, input-space perturbation norms are not applicable because the test vector is unchanged.

Input-level evasion defenses are evaluated on the stored attack-specific samples, whereas model-level defenses are evaluated against newly generated adaptive `TabularScore` attacks. Their rows therefore answer related but not identical robustness questions and should not be compared as if they came from the same attack set.

In [20]:
defense_matrix_df = build_defense_comparison_matrix(
    attack_examples=attack_examples,
    data_poisoning_examples=data_poisoning_examples,
    model_poisoning_examples=model_poisoning_examples,
    best_models=best_models,
    optimal_thresholds=optimal_thresholds,
    adversarially_trained_models=adversarially_trained_models,
    ensemble_adversarial_models=ensemble_adversarial_models,
    adversarial_training_sets=adversarial_training_sets,
    ensemble_adversarial_training_sets=(
        ensemble_adversarial_training_sets
    ),
    X_test=X_test,
    y_test=y_test,
    perturbable_cols=PERTURBABLE_COLS,
    perturbable_idx=PERTURBABLE_IDX,
    train_std=TRAIN_STD,
    train_min=TRAIN_MIN,
    train_max=TRAIN_MAX,
    outlier_detector=outlier_detector,
    evaluate_model_func=evaluate_model,
    seed=SEED + 1000,
)

defense_matrix_display = defense_matrix_df[
    [
        "Model",
        "Threat Type",
        "Attack",
        "Defense",
        "Metric Scope",
        "Rejection Rate",
        "Coverage",
        "Clean Correct N",
        "Clean Correct Positives",
        "Attack Success Rate",
        "Positive Attack Success",
        "Recall",
        "Precision",
        "F2",
        "PR-AUC",
    ]
].sort_values(
    [
        "Threat Type",
        "Model",
        "Attack",
        "Defense",
    ]
)

display(
    defense_matrix_display.style
    .hide(axis="index")
    .format(
        {
            "Rejection Rate": "{:.4f}",
            "Coverage": "{:.4f}",
            "Attack Success Rate": "{:.4f}",
            "Positive Attack Success": "{:.4f}",
            "Recall": "{:.4f}",
            "Precision": "{:.4f}",
            "F2": "{:.4f}",
            "PR-AUC": "{:.4f}",
        },
        na_rep="—",
    )
)

Model,Threat Type,Attack,Defense,Metric Scope,Rejection Rate,Coverage,Clean Correct N,Clean Correct Positives,Attack Success Rate,Positive Attack Success,Recall,Precision,F2,PR-AUC
Decision Tree,Data poisoning,Positive → negative / 1%,Adversarial training,full,—,—,265,27,0.0000,0.0000,0.6279,0.1098,0.3230,0.1725
Decision Tree,Data poisoning,Positive → negative / 1%,Ensemble adversarial training,full,—,—,277,27,0.0000,0.0000,0.6279,0.1154,0.3325,0.1657
Decision Tree,Data poisoning,Positive → negative / 1%,Feature squeezing,full,—,—,265,27,0.0000,0.0000,0.6279,0.1134,0.3293,0.1628
Decision Tree,Data poisoning,Positive → negative / 1%,Isolation Forest rejection,accepted only,0.0360,0.9640,256,27,0.0000,0.0000,0.6279,0.1169,0.3350,0.1668
Decision Tree,Data poisoning,Positive → negative / 1%,None,full,—,—,265,27,0.0000,0.0000,0.6279,0.1130,0.3285,0.1626
Decision Tree,Data poisoning,Positive → negative / 1%,Randomized smoothing,full,—,—,265,27,0.0000,0.0000,0.6512,0.1167,0.3398,0.1627
Decision Tree,Data poisoning,Positive → negative / 10%,Adversarial training,full,—,—,265,27,0.0075,0.0000,0.6744,0.1174,0.3461,0.1647
Decision Tree,Data poisoning,Positive → negative / 10%,Ensemble adversarial training,full,—,—,277,27,0.0181,0.0000,0.6512,0.1172,0.3406,0.1654
Decision Tree,Data poisoning,Positive → negative / 10%,Feature squeezing,full,—,—,265,27,0.0792,0.0000,0.6744,0.1094,0.3318,0.1773
Decision Tree,Data poisoning,Positive → negative / 10%,Isolation Forest rejection,accepted only,0.0360,0.9640,256,27,0.0742,0.0000,0.6744,0.1142,0.3404,0.1810


The unified matrix confirms that defense effectiveness is **threat-specific**.

For **evasion**, input transformations can reduce attack success in several cases. Against Decision Tree + HopSkipJump, feature squeezing reduces the overall attack success rate from **0.7698 to 0.7132**, and randomized smoothing reduces it further to **0.6830**; neither removes the vulnerability to positive-class attacks, which remains **0.1481**. For XGBoost + TabularScore, both feature squeezing and randomized smoothing reduce attack success from **0.0068 to 0.0000** on the evaluated subset. By contrast, Isolation Forest does not consistently improve the attack metrics and necessarily reduces coverage; for example, coverage is **0.8830** for Decision Tree + HopSkipJump.

For **data poisoning**, input-only defenses usually change little because the attack has already modified the learned model. A clear example is MLP + 5% targeted positive-to-negative poisoning: the undefended, feature-squeezed, randomized-smoothed, and Isolation-Forest rows all have **0.0000 recall** and **1.0000 positive attack success**. Model-level retraining provides partial recovery: adversarial training raises recall to **0.0930**, while ensemble adversarial training reaches **0.1628**, although positive attack success remains high (**0.7273** and **0.6154**).

A similar pattern appears for **model poisoning**. At severity 0.50, Decision Tree and XGBoost still reach **0 recall** under the evaluated defenses. Input transformations cannot undo maliciously changed model parameters. Model-level defenses improve average behavior at lower severities, but they do not eliminate the worst-case high-severity failure.

The matrix therefore supports two main conclusions: **input preprocessing is primarily useful against input-space evasion**, while **training-based hardening can improve some poisoning scenarios but cannot guarantee protection against direct model tampering**.

The comparison between input defenses and adversarial-training rows should remain qualitative for evasion, because the latter are evaluated against fresh adaptive `TabularScore` attacks rather than exactly the same attack set.

## 6. Robustness Scorecard

The scorecard aggregates results **within each threat family** rather than combining fundamentally different attacks into one global score.

For full-scope evaluations:

$$
\text{Recall Retention}
=
\frac{\text{Recall under attack}}
{\text{clean-test recall}}
$$

and

$$
\text{Recall Drop}
=
\text{clean-test recall}
-
\text{recall under attack}.
$$

Isolation-Forest rows retain conditional performance and coverage, but recall retention is left undefined because their classification metrics are computed only on accepted observations.

For evasion, recall is measured on attack subsets selected from initially correct observations. Consequently, recall-retention values can exceed 1 when divided by the model's full-test recall. Such values **must not be interpreted as an actual improvement over clean performance**; they mainly reflect the conditional construction of the attack set.


In [21]:
clean_recall = {}

for model_name, model in best_models.items():
    threshold = float(optimal_thresholds[model_name])
    metrics = evaluate_model(
        model,
        threshold,
        X_test,
        y_test,
    )
    clean_recall[model_name] = metrics["Recall"]

scorecard = defense_matrix_df.copy()

scorecard["Clean Recall"] = (
    scorecard["Model"].map(clean_recall)
)

full_scope = scorecard["Metric Scope"].eq("full")

scorecard["Recall Retention"] = np.where(
    full_scope,
    (
        scorecard["Recall"]
        / scorecard["Clean Recall"].replace(0, np.nan)
    ),
    np.nan,
)
scorecard["Recall Drop"] = np.where(
    full_scope,
    scorecard["Clean Recall"] - scorecard["Recall"],
    np.nan,
)

scorecard_summary = (
    scorecard
    .groupby(
        [
            "Model",
            "Threat Type",
            "Defense",
            "Metric Scope",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        Mean_Recall=("Recall", "mean"),
        Mean_Precision=("Precision", "mean"),
        Mean_F2=("F2", "mean"),
        Mean_PR_AUC=("PR-AUC", "mean"),
        Mean_Attack_Success=(
            "Attack Success Rate",
            "mean",
        ),
        Mean_Positive_Attack_Success=(
            "Positive Attack Success",
            "mean",
        ),
        Mean_Coverage=("Coverage", "mean"),
        Mean_Recall_Retention=(
            "Recall Retention",
            "mean",
        ),
        Worst_Case_Recall_Retention=(
            "Recall Retention",
            "min",
        ),
        Mean_Recall_Drop=(
            "Recall Drop",
            "mean",
        ),
    )
    .sort_values(
        [
            "Threat Type",
            "Model",
            "Mean_Positive_Attack_Success",
        ],
        ascending=[
            True,
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)

display(
    scorecard_summary.style
    .hide(axis="index")
    .format(
        {
            "Mean_Recall": "{:.4f}",
            "Mean_Precision": "{:.4f}",
            "Mean_F2": "{:.4f}",
            "Mean_PR_AUC": "{:.4f}",
            "Mean_Attack_Success": "{:.4f}",
            "Mean_Positive_Attack_Success": "{:.4f}",
            "Mean_Coverage": "{:.4f}",
            "Mean_Recall_Retention": "{:.4f}",
            "Worst_Case_Recall_Retention": "{:.4f}",
            "Mean_Recall_Drop": "{:.4f}",
        },
        na_rep="—",
    )
)

Model,Threat Type,Defense,Metric Scope,Mean_Recall,Mean_Precision,Mean_F2,Mean_PR_AUC,Mean_Attack_Success,Mean_Positive_Attack_Success,Mean_Coverage,Mean_Recall_Retention,Worst_Case_Recall_Retention,Mean_Recall_Drop
Decision Tree,Data poisoning,Ensemble adversarial training,full,0.6473,0.1172,0.3399,0.1778,0.0259,0.0000,—,0.9991,0.9692,0.0006
Decision Tree,Data poisoning,Adversarial training,full,0.6395,0.1148,0.3340,0.1677,0.0107,0.0062,—,0.9872,0.9333,0.0083
Decision Tree,Data poisoning,Randomized smoothing,full,0.6434,0.1149,0.3349,0.1635,0.0327,0.0123,—,0.9931,0.9692,0.0044
Decision Tree,Data poisoning,Feature squeezing,full,0.6357,0.1137,0.3311,0.1630,0.0340,0.0185,—,0.9812,0.9333,0.0122
Decision Tree,Data poisoning,Isolation Forest rejection,accepted only,0.6357,0.1184,0.3390,0.1678,0.0312,0.0185,0.9640,—,—,—
Decision Tree,Data poisoning,None,full,0.6357,0.1136,0.3310,0.1630,0.0340,0.0185,—,0.9812,0.9333,0.0122
MLP,Data poisoning,Feature squeezing,full,0.2248,0.1786,0.2126,0.1926,0.0590,0.2083,—,2.2094,0.0000,-0.1231
MLP,Data poisoning,Isolation Forest rejection,accepted only,0.2287,0.1882,0.2182,0.2004,0.0594,0.2083,0.9640,—,—,—
MLP,Data poisoning,None,full,0.2287,0.1822,0.2165,0.1929,0.0586,0.2083,—,2.2475,0.0000,-0.1269
MLP,Data poisoning,Randomized smoothing,full,0.2287,0.1797,0.2157,0.1927,0.0598,0.2083,—,2.2475,0.0000,-0.1269


The scorecard makes the defense trade-offs clearer by averaging within each model and threat family.

For **Decision Tree data poisoning**, ensemble adversarial training performs best on positive-class preservation: mean positive attack success is **0.0000**, mean recall is **0.6473**, and worst-case recall retention is **0.9692**. Standard adversarial training is also strong, with **0.0062** mean positive attack success. The input-only defenses remain much closer to the undefended baseline, confirming that they do not address corrupted training labels directly.

For **XGBoost data poisoning**, ensemble adversarial training again has the lowest mean positive attack success (**0.0298**) and a worst-case recall retention of **0.9178**. Its mean recall (**0.7481**) is higher than the clean-test recall because several poisoning configurations shift the fixed operating point toward predicting more positives; this should not be interpreted as improved model quality without considering precision and F2.

The **MLP data-poisoning scorecard is less straightforward**. Input-level defenses have lower mean positive attack success (**0.2083**) than adversarial training (**0.3182**) or ensemble adversarial training (**0.2692**), but their worst-case recall retention is **0.0000**, reflecting the complete collapse under the 5% targeted attack. Ensemble adversarial training provides a much stronger worst-case retention (**1.5999** relative to the very low clean MLP recall), although the ratio is inflated by the weak clean baseline.

For **model poisoning**, ensemble adversarial training provides the best mean recall for every model: **0.2791** for the Decision Tree, **0.2403** for XGBoost, and **0.1938** for the MLP, versus undefended means of **0.2171**, **0.1705**, and **0.0698**. It also lowers mean positive attack success for all three models. Nevertheless, worst-case recall retention remains **0.0000 for Decision Tree and XGBoost**, because severity 0.50 still eliminates all positive detections.

For **evasion**, the scorecard must be read with extra caution. Recall-retention values above 1 arise because attack evaluation is conditioned on initially correct subsets and then divided by full-test recall. They are therefore not evidence that the attack or defense improves the original model. Within the directly comparable input-defense rows, randomized smoothing and feature squeezing reduce attack success for some model/attack pairs, while XGBoost remains the most robust base architecture.

Overall, the scorecard favors **ensemble adversarial training for poisoning resilience**, but no single defense dominates every model and threat family.

## 7. Attack Vulnerability Ranking

The ranking now includes the **undefended** scenarios from all three threat families.

`Positive Attack Success` remains the primary ranking criterion because the positive class corresponds to hospital readmission. `Attack Success Rate` is used as the broader secondary measure.

Perturbation norms are reported only for evasion attacks; they are not applicable to data/model poisoning because the test input does not change.

In [22]:
attack_ranking_df = (
    defense_matrix_df[
        (
            defense_matrix_df["Defense"] == "None"
        )
        & (
            defense_matrix_df["Metric Scope"] == "full"
        )
    ][
        [
            "Model",
            "Threat Type",
            "Attack",
            "Attack Success Rate",
            "Positive Attack Success",
            "Recall",
            "Mean L2",
            "Mean Linf",
        ]
    ]
    .sort_values(
        [
            "Positive Attack Success",
            "Attack Success Rate",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

display(
    attack_ranking_df.style
    .hide(axis="index")
    .format(
        {
            "Attack Success Rate": "{:.4f}",
            "Positive Attack Success": "{:.4f}",
            "Recall": "{:.4f}",
            "Mean L2": "{:.4f}",
            "Mean Linf": "{:.4f}",
        },
        na_rep="—",
    )
)

Model,Threat Type,Attack,Attack Success Rate,Positive Attack Success,Recall,Mean L2,Mean Linf
Decision Tree,Model poisoning,Artifact tampering / severity=0.50,0.1019,1.0000,0.0000,—,—
XGBoost,Model poisoning,Artifact tampering / severity=0.50,0.0959,1.0000,0.0000,—,—
MLP,Data poisoning,Positive → negative / 5%,0.0177,1.0000,0.0000,—,—
MLP,Model poisoning,Artifact tampering / severity=0.50,0.0177,1.0000,0.0000,—,—
XGBoost,Model poisoning,Artifact tampering / severity=0.25,0.0719,0.7500,0.1628,—,—
Decision Tree,Model poisoning,Artifact tampering / severity=0.25,0.0642,0.6296,0.2326,—,—
MLP,Model poisoning,Artifact tampering / severity=0.25,0.0111,0.6250,0.0698,—,—
XGBoost,Model poisoning,Artifact tampering / severity=0.10,0.0445,0.4643,0.3488,—,—
Decision Tree,Model poisoning,Artifact tampering / severity=0.10,0.0340,0.3333,0.4186,—,—
XGBoost,Data poisoning,Positive → negative / 10%,0.0274,0.2857,0.4651,—,—


The ranking shows that the most severe **model-artifact poisoning scenarios dominate the positive-class risk**. At severity **0.50**, Decision Tree and XGBoost both have **1.0000 positive attack success and 0.0000 recall**. The MLP also reaches **1.0000 positive attack success** under severity-0.50 model poisoning.

The **MLP with 5% targeted positive-to-negative data poisoning** is similarly critical: it records **1.0000 positive attack success and 0.0000 recall** on the fixed poisoning evaluation subset. This highlights that a relatively small targeted corruption can be more damaging than a larger untargeted corruption for a model with an already fragile positive-class operating point.

At severity 0.25, model poisoning remains severe: positive attack success is **0.7500 for XGBoost**, **0.6296 for the Decision Tree**, and **0.6250 for the MLP**. These values are substantially larger than most evasion results.

Among evasion attacks, **MLP + TabularScore** has the highest positive attack success (**0.2500**), although it is based on only eight initially correct positives. **Decision Tree + HopSkipJump** follows at **0.1481** and has a very high overall attack success rate of **0.7698**. XGBoost's strongest evasion result is much smaller: TabularScore reaches **0.0714 positive attack success** and **0.0068 overall attack success**.

The ranking therefore indicates that **direct compromise of the model artifact is the most consistently destructive threat in this experiment**, while targeted label poisoning can also create severe model-specific failures.

## 8. Conclusion

The central finding is that **robustness is stage-specific**. A defense designed to sanitize an input cannot repair a compromised training set or fitted artifact, while training-based hardening does not guarantee protection against direct parameter tampering. In a readmission setting, the most informative robustness indicators are therefore the combination of **positive attack success, recall, F2, PR-AUC, and coverage where rejection is used**, rather than a single aggregate robustness score.

Within the experiments performed here, the **Decision Tree offers the best clean recall**, **XGBoost provides the strongest resistance to the tested evasion attacks**, and **high-severity model-artifact poisoning represents the most serious common vulnerability**. Ensemble adversarial training improves resilience to several poisoning configurations, but the results do not support treating any of the evaluated defenses as a complete solution.